# 12 — V4 Unified DEV Feature Generation + Preregistered 3-Architecture Tournament

**Purpose:** Generate astrology features for the frozen 101-pair unified development corpus for the first time, then run only the 3 preregistered winner-eligible architectures against Production Control and chronology/lifecycle nuisance baselines.

**Hard rules**
- Do not alter the 101 pairs or axis labels.
- Do not add architectures after seeing results.
- Do not hand-tune feature weights.
- Do not open NEW_CONFIRM / Validation_B / Public_CHECK / Public_FINAL.
- `NUISANCE_AGE_AXIS` retains the pre-existing V4 age × axis interaction design, so axis-specific chronology is explicitly controlled.


In [4]:

from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import hashlib, json, math, re, sys, warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_V4_UNIFIED_DEV_3ARCH_TOURNAMENT_20260816"
SEED = 20260816
OUTER_REPEATS = 10
OUTER_FOLDS = 5
INNER_FOLDS = 3
N_BOOTSTRAP = 20000

ELASTIC_C_GRID = [0.03, 0.10, 0.30, 1.00]
ELASTIC_L1_GRID = [0.25, 0.50, 0.75, 1.00]

MIN_OVERALL_MACRO = 0.55
MIN_DELTA_VS_REFERENCE = 0.01
MIN_BOOT_P_DELTA_GT_0 = 0.80
MIN_P10 = 0.48

WINNER_ELIGIBLE = [
    "TG10_STEM_BRANCH_L2",
    "ALL_L2",
    "ALL_ELASTICNET_AUTO",
]

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError("Run inside the Chartpalja saju repo (saju_engine.py required).")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().map({"true": True, "false": False}).fillna(False)

ROOT = find_repo_root()

PAIR_PATH = ROOT / "research/ml/artifacts/v4_unified_dev_wave2/V4_UNIFIED_DEV_COMBINED_FROZEN_PAIRS.csv"
FREEZE_PATH = ROOT / "research/ml/artifacts/v4_unified_dev_wave2/V4_UNIFIED_DEV_WAVE2_FREEZE_DECISION.json"

PREREG_DIR = ROOT / "research/ml/artifacts/v4_unified_dev_modeling_preregistration"
PREREG_DECISION_PATH = PREREG_DIR / "V4_COMBINED_FEATURE_PREREGISTRATION_DECISION.json"
PREREG_LINEAGE_PATH = PREREG_DIR / "V4_COMBINED_MODELING_PREREGISTRATION_LINEAGE.json"
SPEC_PATH = ROOT / "research/ml_corpus/v4_unified_dev_modeling/V4_UNIFIED_DEV_COMBINED_MODELING_PREREGISTRATION.json"

W1_ROSTER = ROOT / "research/ml/artifacts/v4_unified_dev_roster/V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv"
W2_ROSTER = ROOT / "research/ml/artifacts/v4_unified_dev_wave2_roster/V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv"

ARTIFACT_DIR = ROOT / "research/ml/artifacts/v4_unified_dev_3arch_tournament"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PAIR_PATH, FREEZE_PATH, PREREG_DECISION_PATH, PREREG_LINEAGE_PATH, SPEC_PATH, W1_ROSTER, W2_ROSTER]:
    if not p.exists():
        raise FileNotFoundError(p)

pairs = pd.read_csv(PAIR_PATH)
w1_roster = pd.read_csv(W1_ROSTER)
w2_roster = pd.read_csv(W2_ROSTER)

with open(FREEZE_PATH, encoding="utf-8") as f:
    freeze = json.load(f)
with open(PREREG_DECISION_PATH, encoding="utf-8") as f:
    prereg = json.load(f)
with open(PREREG_LINEAGE_PATH, encoding="utf-8") as f:
    lineage = json.load(f)
with open(SPEC_PATH, encoding="utf-8") as f:
    spec = json.load(f)

print("="*90)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("repo:", ROOT)
print("output:", ARTIFACT_DIR)
print("="*90)


NOTEBOOK VERSION: SAJU_ML_V4_UNIFIED_DEV_3ARCH_TOURNAMENT_20260816
repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_unified_dev_3arch_tournament


## 1. Immutable preflight before first astrology feature generation

In [5]:

assert freeze["status"] == "V4_UNIFIED_DEV_WAVE2_FROZEN_COMBINED_TARGETS_MET_READY_FOR_FEATURE_GENERATION_GATE"
assert prereg["status"] == "V4_COMBINED_FEATURE_PREREGISTERED_READY_FOR_FEATURE_GENERATION"

assert len(pairs) == 101
assert pairs.subject_id.nunique() == 101
assert pairs.pair_id.nunique() == 101
assert (pairs.positive_year.astype(int) != pairs.negative_year.astype(int)).all()
assert pairs.groupby("axis").size().to_dict() == {"COMPETITIVE":31, "PROJECT":25, "STATUS":45}

if "astrology_scored" in pairs.columns:
    assert as_bool(pairs["astrology_scored"]).sum() == 0

assert lineage["pair_sha256"] == sha256_file(PAIR_PATH)
assert lineage["preregistration_spec_sha256"] == sha256_file(SPEC_PATH)
assert lineage["winner_eligible_architectures"] == WINNER_ELIGIBLE
assert prereg["winner_eligible_architectures"] == WINNER_ELIGIBLE

engine_path = ROOT / "saju_engine.py"
assert sha256_file(engine_path) == lineage["saju_engine_sha256_at_preregistration"], (
    "saju_engine.py changed after preregistration. Stop: feature lineage is no longer frozen."
)

assert all(v is False for v in prereg["holdout_integrity"].values())
assert all(v is False for v in freeze["holdout_integrity"].values())

pairs = pairs.copy()
pairs["positive_earlier_calc"] = pairs.positive_year.astype(int) < pairs.negative_year.astype(int)
pairs["abs_year_gap"] = (pairs.positive_year.astype(int) - pairs.negative_year.astype(int)).abs()
pairs["subject_equal_pair_weight"] = 1.0

print("Immutable preflight: PASS")
print("positive-earlier overall:", float(pairs.positive_earlier_calc.mean()))
display(
    pairs.groupby("axis")
    .agg(n=("pair_id","size"), positive_earlier=("positive_earlier_calc","mean"),
         median_gap=("abs_year_gap","median"))
)


Immutable preflight: PASS
positive-earlier overall: 0.5643564356435643


,n,positive_earlier,median_gap
axis,,,
COMPETITIVE,31,0.580645,1.0
PROJECT,25,0.200000,3.0
STATUS,45,0.755556,7.0


### Chronology note

The combined corpus is balanced overall but not inside every axis. This is **not** repaired by changing membership. Instead, the pre-existing `NUISANCE_AGE_AXIS` baseline explicitly includes age-delta × axis interactions and is fitted only inside each outer-training split. Any astrology winner must beat this nuisance baseline.


## 2. Canonical engine / astrology feature helpers

In [6]:
sys.path.insert(0, str(ROOT))

import saju_engine as se
import sajupy

STEMS = list("甲乙丙丁戊己庚辛壬癸")
BRANCHES = list("子丑寅卯辰巳午未申酉戌亥")
ELEMENTS = ["wood", "fire", "earth", "metal", "water"]

STEM_ELEMENT_EN = {
    "甲":"wood","乙":"wood","丙":"fire","丁":"fire","戊":"earth",
    "己":"earth","庚":"metal","辛":"metal","壬":"water","癸":"water"
}
STEM_YINYANG = {
    "甲":"yang","乙":"yin","丙":"yang","丁":"yin","戊":"yang",
    "己":"yin","庚":"yang","辛":"yin","壬":"yang","癸":"yin"
}
BRANCH_ELEMENT_EN = {
    "子":"water","丑":"earth","寅":"wood","卯":"wood","辰":"earth","巳":"fire",
    "午":"fire","未":"earth","申":"metal","酉":"metal","戌":"earth","亥":"water"
}
HIDDEN_STEMS = {
    "子":["癸"], "丑":["己","癸","辛"], "寅":["甲","丙","戊"], "卯":["乙"],
    "辰":["戊","乙","癸"], "巳":["丙","戊","庚"], "午":["丁","己"],
    "未":["己","丁","乙"], "申":["庚","壬","戊"], "酉":["辛"],
    "戌":["戊","辛","丁"], "亥":["壬","甲"]
}

GENERATES = {
    "wood":"fire","fire":"earth","earth":"metal","metal":"water","water":"wood"
}
CONTROLS = {
    "wood":"earth","earth":"water","water":"fire","fire":"metal","metal":"wood"
}

STEM_COMBINES = {
    frozenset(x)
    for x in [("甲","己"),("乙","庚"),("丙","辛"),("丁","壬"),("戊","癸")]
}
STEM_CLASHES = {
    frozenset(x)
    for x in [("甲","庚"),("乙","辛"),("丙","壬"),("丁","癸")]
}
BRANCH_COMBINES = {
    frozenset(x)
    for x in [("子","丑"),("寅","亥"),("卯","戌"),("辰","酉"),("巳","申"),("午","未")]
}
BRANCH_CLASHES = {
    frozenset(x)
    for x in [("子","午"),("丑","未"),("寅","申"),("卯","酉"),("辰","戌"),("巳","亥")]
}
BRANCH_HARMS = {
    frozenset(x)
    for x in [("子","未"),("丑","午"),("寅","巳"),("卯","辰"),("申","亥"),("酉","戌")]
}
BRANCH_BREAKS = {
    frozenset(x)
    for x in [("子","酉"),("丑","辰"),("寅","亥"),("卯","午"),("巳","申"),("未","戌")]
}
SELF_PUNISH = set(["辰","午","酉","亥"])
PUNISH_TRIPLES = [set("寅巳申"), set("丑未戌")]

UNSEONG_STATES = [
    "장생","목욕","관대","건록","제왕","쇠",
    "병","사","묘","절","태","양"
]
UNSEONG_START = {
    "甲":"亥", "乙":"午",
    "丙":"寅", "丁":"酉",
    "戊":"寅", "己":"酉",
    "庚":"巳", "辛":"子",
    "壬":"申", "癸":"卯",
}
YANG_STEMS = set("甲丙戊庚壬")
SHINSAL_KEYWORDS = ("신살", "귀인", "공망")

def parse_utc_offset(raw):
    if isinstance(raw, (int, float)):
        return float(raw)
    s = str(raw).strip()
    sign = -1.0 if s.startswith("-") else 1.0
    s = s[1:] if s[:1] in "+-" else s
    hh, mm = s.split(":")
    return sign * (int(hh) + int(mm) / 60.0)

def public_lon_corrected_birth(public_birth):
    y, m, d = map(int, public_birth["date"].split("-"))
    hh, mi = map(int, public_birth["time"].split(":")[:2])
    lon = float(public_birth["longitude"])
    utc_hours = parse_utc_offset(public_birth["utc_offset"])

    calc = sajupy.get_saju_calculator()
    civil = datetime(y, m, d, hh, mi)

    corr = float(
        calc._calculate_solar_time_correction(lon, utc_hours)
    )
    h2, min2, dc = calc._adjust_time_for_solar(
        civil.hour, civil.minute, corr
    )
    y2, m2, d2 = calc._adjust_date_for_solar(
        civil.year, civil.month, civil.day, dc
    )

    return {
        "calendar": public_birth.get("calendar", "solar"),
        "y": y2, "m": m2, "d": d2,
        "h": h2, "min": min2,
    }

def pillar_chars(value):
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        a, b = str(value[0]), str(value[1])
        if a in STEMS and b in BRANCHES:
            return a, b

    s = str(value)
    stem = next((c for c in s if c in STEMS), None)
    branch = next((c for c in s if c in BRANCHES), None)
    return stem, branch

def sexagenary_year_pillar(year):
    i = (int(year) - 1984) % 60
    return STEMS[i % 10] + BRANCHES[i % 12]

def ten_god_group(day_stem, other_stem):
    de = STEM_ELEMENT_EN[day_stem]
    oe = STEM_ELEMENT_EN[other_stem]

    if de == oe:
        return "peer"
    if GENERATES[de] == oe:
        return "output"
    if CONTROLS[de] == oe:
        return "wealth"
    if CONTROLS[oe] == de:
        return "officer"
    if GENERATES[oe] == de:
        return "resource"

    raise RuntimeError((day_stem, other_stem))

def onehot(out, prefix, value, vocab):
    for v in vocab:
        out[prefix + str(v)] = float(value == v)

def pair_relation(a, b, relation_set):
    if a is None or b is None:
        return 0.0
    return float(frozenset((a, b)) in relation_set)

def branch_punishment(a, b):
    if a is None or b is None:
        return 0.0
    if a == b and a in SELF_PUNISH:
        return 1.0
    if set([a, b]) == set(["子", "卯"]):
        return 1.0
    return float(any(set([a, b]).issubset(x) for x in PUNISH_TRIPLES))

def get_strength_label(result):
    x = result.get("신강신약")
    if isinstance(x, dict):
        return x.get("판정") or x.get("label")
    if isinstance(x, str):
        return x
    return None

def get_daewoon_pillar(meta_row):
    for key in ("대운_pillar", "대운", "daewoon_pillar"):
        if meta_row.get(key):
            st, br = pillar_chars(meta_row[key])
            if st and br:
                return st + br
    return None

def get_daewoon_row(daewoon, year):
    for row in daewoon:
        if int(row["start_year"]) <= int(year) < int(row["end_year"]):
            return row
    return None

def get_sewoon_pillar(meta_row, year):
    for key in ("세운_pillar", "세운", "연주", "year_pillar"):
        if meta_row.get(key):
            st, br = pillar_chars(meta_row[key])
            if st and br:
                return st + br
    return sexagenary_year_pillar(year)

def control_score_from_meta(meta_row):
    candle = meta_row.get("candle") or {}
    value = candle.get("close")
    return float(value) if value is not None else np.nan

def local_twelve_unseong(day_stem, branch):
    start = UNSEONG_START[day_stem]
    start_i = BRANCHES.index(start)
    branch_i = BRANCHES.index(branch)
    direction = 1 if day_stem in YANG_STEMS else -1
    steps = ((branch_i - start_i) * direction) % 12
    return UNSEONG_STATES[steps]

def get_unseong_state(day_stem, branch):
    for fn_name in ("twelve_unseong", "_twelve_unseong"):
        fn = getattr(se, fn_name, None)
        if fn is None:
            continue

        for args in ((day_stem, branch), (branch, day_stem)):
            try:
                value = fn(*args)
                if isinstance(value, str) and value in UNSEONG_STATES:
                    return value
            except Exception:
                pass

    return local_twelve_unseong(day_stem, branch)

def strength_bucket(result):
    x = result.get("신강신약") or {}
    verdict = x.get("판정") if isinstance(x, dict) else str(x)
    verdict = verdict or ""

    if any(k in verdict for k in ("신강", "태강", "극왕")):
        return "strong"
    if any(k in verdict for k in ("신약", "태약", "극약")):
        return "weak"
    return "neutral"

# V1.3 Yongshin used the engine's element labels.
STEM_ELEMENT_ENGINE = getattr(se, "STEM_ELEMENT", {
    "甲":"목","乙":"목","丙":"화","丁":"화","戊":"토",
    "己":"토","庚":"금","辛":"금","壬":"수","癸":"수"
})
BRANCH_ELEMENT_ENGINE = getattr(se, "BRANCH_ELEMENT_MAIN", {
    "子":"수","丑":"토","寅":"목","卯":"목","辰":"토","巳":"화",
    "午":"화","未":"토","申":"금","酉":"금","戌":"토","亥":"수"
})

def yongshin_block_features(result, stem, branch, prefix):
    out = {}
    yong = result.get("용신") or {}
    day_stem = (result.get("원국") or {}).get("day", ["", ""])[0]

    fit = {}

    if yong and day_stem and hasattr(se, "_check_yongshin_fit"):
        try:
            fit = se._check_yongshin_fit(
                stem, branch, yong, day_stem
            ) or {}
        except Exception:
            fit = {}

    mapping = [
        ("용신부합", "yong_fit"),
        ("희신부합", "hee_fit"),
        ("기신부합", "gi_fit"),
        ("구신부합", "gu_fit"),
        ("용신부합_천간", "yong_stem_fit"),
        ("용신부합_지지", "yong_branch_fit"),
    ]

    for source_key, target_key in mapping:
        out[
            "yongshin__%s_%s" % (prefix, target_key)
        ] = float(fit.get(source_key) or 0.0)

    yong_e = yong.get("용신_오행") or ""
    hee = set(yong.get("희신_오행") or [])
    gi = set(yong.get("기신_오행") or [])
    gu = set(yong.get("구신_오행") or [])

    fav = (set([yong_e]) | hee) - set([""])
    unfav = (gi | gu) - set([""])

    stem_e = STEM_ELEMENT_ENGINE.get(stem, "")
    branch_e = BRANCH_ELEMENT_ENGINE.get(branch, "")

    out["yongshin__%s_supplies_fav" % prefix] = float(
        stem_e in fav or branch_e in fav
    )
    out["yongshin__%s_supplies_unfav" % prefix] = float(
        stem_e in unfav or branch_e in unfav
    )

    return out

def unseong_block_features(result, branch, prefix):
    out = {}
    day_stem = (result.get("원국") or {}).get("day", ["", ""])[0]
    state = get_unseong_state(day_stem, branch)

    for state_name in UNSEONG_STATES:
        out[
            "unseong__%s_state_%s" % (prefix, state_name)
        ] = float(state == state_name)

    raw_map = getattr(se, "_UNSEONG_SCORE", {})
    if isinstance(raw_map, dict):
        raw_score = float(raw_map.get(state, 0.0))
    else:
        raw_score = 0.0

    out["unseong__%s_raw_score" % prefix] = raw_score

    bucket = strength_bucket(result)
    sign = 1.0 if bucket == "weak" else (
        -1.0 if bucket == "strong" else 0.0
    )

    out["unseong__%s_score_x_strength" % prefix] = (
        raw_score * sign
    )

    return out

def safe_name(value):
    return re.sub(
        r"[^0-9A-Za-z가-힣_]+",
        "_",
        str(value),
    )[:120]

def flatten_all(value, path, out):
    key = safe_name(path)

    if isinstance(value, bool):
        out["shinsal__" + key] = float(value)

    elif isinstance(value, (int, float)) and np.isfinite(value):
        out["shinsal__" + key] = float(value)

    elif isinstance(value, str):
        if value.strip():
            out[
                "shinsal__%s__%s" % (key, safe_name(value))
            ] = 1.0

    elif isinstance(value, dict):
        for k2, v2 in value.items():
            flatten_all(v2, path + "." + str(k2), out)

    elif isinstance(value, (list, tuple, set)):
        out["shinsal__%s__count" % key] = float(len(value))

        for item in value:
            if isinstance(item, str):
                out[
                    "shinsal__%s__%s" % (key, safe_name(item))
                ] = 1.0
            elif isinstance(item, dict):
                flatten_all(item, path, out)

def flatten_selected(obj, prefix="", out=None):
    if out is None:
        out = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = prefix + "." + str(key) if prefix else str(key)

            if any(token in path for token in SHINSAL_KEYWORDS):
                flatten_all(value, path, out)

            elif isinstance(value, (dict, list, tuple)):
                flatten_selected(value, path, out)

    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            flatten_selected(
                value,
                "%s[%d]" % (prefix, i),
                out,
            )

    return out

def shinsal_features(dw_row, meta_row):
    out = {}

    if dw_row:
        breakdown = dw_row.get("breakdown") or {}

        if "shinsal" in breakdown:
            out["shinsal__dw_breakdown_scalar"] = float(
                breakdown.get("shinsal") or 0.0
            )

        gongmang = dw_row.get("gongmang_factors") or {}

        for key, value in gongmang.items():
            if isinstance(value, (int, float)):
                out[
                    "shinsal__dw_gongmang_%s" % safe_name(key)
                ] = float(value)

        indicators = dw_row.get("indicators") or {}
        noble = indicators.get("귀인력")

        if isinstance(noble, (int, float)):
            out["shinsal__dw_noble_power"] = float(noble)

        flatten_selected(dw_row, "dw", out)

    flatten_selected(meta_row, "sw", out)
    return out

In [7]:
EXACT_TENGODS = [
    "bijian", "jiecai", "shishen", "shangguan",
    "pian_cai", "zheng_cai", "qi_sha", "zheng_guan",
    "pian_yin", "zheng_yin",
]

def exact_ten_god(day_stem, other_stem):
    de = STEM_ELEMENT_EN[day_stem]
    oe = STEM_ELEMENT_EN[other_stem]
    same_polarity = STEM_YINYANG[day_stem] == STEM_YINYANG[other_stem]

    if de == oe:
        return "bijian" if same_polarity else "jiecai"
    if GENERATES[de] == oe:
        return "shishen" if same_polarity else "shangguan"
    if CONTROLS[de] == oe:
        return "pian_cai" if same_polarity else "zheng_cai"
    if CONTROLS[oe] == de:
        return "qi_sha" if same_polarity else "zheng_guan"
    if GENERATES[oe] == de:
        return "pian_yin" if same_polarity else "zheng_yin"
    raise RuntimeError((day_stem, other_stem))

def exact_tg_distribution(day_stem, stems):
    counts = Counter(exact_ten_god(day_stem, st) for st in stems)
    n = float(max(1, len(stems)))
    return {tg: counts[tg] / n for tg in EXACT_TENGODS}

## 3. Build subject map from the two frozen rosters

In [8]:

all_roster = pd.concat([w1_roster, w2_roster], ignore_index=True, sort=False)
assert all_roster.subject_id.nunique() == len(all_roster)
assert set(pairs.subject_id).issubset(set(all_roster.subject_id))

subject_map = {}
for _, r in all_roster.iterrows():
    if r.subject_id not in set(pairs.subject_id):
        continue
    subject_map[r.subject_id] = {
        "subject_id": r.subject_id,
        "name": r["name"],
        "gender": r["gender"],
        "birth": {
            "calendar": "solar",
            "date": str(r["birth_date"]),
            "time": str(r["birth_time"]),
            "utc_offset": str(r["utc_offset"]),
            "longitude": float(r["longitude"]),
            "latitude": float(r["latitude"]),
            "place": str(r["birth_place"]),
        }
    }

assert len(subject_map) == 101

birth_year = {
    sid: int(subject["birth"]["date"].split("-")[0])
    for sid, subject in subject_map.items()
}

pairs["positive_age"] = [
    int(y) - birth_year[sid]
    for sid, y in zip(pairs.subject_id, pairs.positive_year)
]
pairs["negative_age"] = [
    int(y) - birth_year[sid]
    for sid, y in zip(pairs.subject_id, pairs.negative_year)
]

assert (pairs.positive_age >= 0).all()
assert (pairs.negative_age >= 0).all()

print("Frozen subject lineage:", len(subject_map))


Frozen subject lineage: 101


## 4. Current-runtime engine adapter

In [9]:

import saju_engine as se
import sajupy

ENGINE_CALCULATOR = sajupy.get_saju_calculator()
ENGINE_MIN_YEAR = int(ENGINE_CALCULATOR.min_year)
ENGINE_MAX_YEAR = int(ENGINE_CALCULATOR.max_year)

def compute_v4_subject_engine(subject):
    birth = subject["birth"]
    corrected = public_lon_corrected_birth(birth)

    if not (ENGINE_MIN_YEAR <= int(corrected["y"]) <= ENGINE_MAX_YEAR):
        raise ValueError((subject["subject_id"], corrected["y"], ENGINE_MIN_YEAR, ENGINE_MAX_YEAR))

    inp = se.BirthInput(
        year=int(corrected["y"]), month=int(corrected["m"]), day=int(corrected["d"]),
        hour=int(corrected["h"]), minute=int(corrected["min"]),
        gender=subject["gender"], calendar=corrected.get("calendar", "solar"),
        is_leap_month=False, use_solar_time=False, utc_offset=9,
    )

    result = se.compute_all(inp)
    daewoon = se.build_daewoon_detail(result)
    timeline = result["chart_data"]["연도별_타임라인"]
    meta = {int(row["year"]): row for row in timeline}
    return result, daewoon, meta


## 5. Frozen V4 feature extractor

In [10]:
def extract_v2_all_features(subject, year, result, daewoon, meta_row):
    natal = result["원국"]

    natal_pillars = {}
    for pos in ("year", "month", "day", "hour"):
        stem, branch = pillar_chars(natal[pos])
        natal_pillars[pos] = (stem, branch)

    day_stem = natal_pillars["day"][0]

    # Preserve V1 base feature semantics by reading the Daewoon pillar
    # from the annual timeline metadata.
    dw_stem, dw_branch = pillar_chars(
        get_daewoon_pillar(meta_row)
    )

    sw_stem, sw_branch = pillar_chars(
        get_sewoon_pillar(meta_row, year)
    )

    if not all([
        day_stem,
        dw_stem,
        dw_branch,
        sw_stem,
        sw_branch,
    ]):
        raise ValueError(
            "Pillar parse failed: %s %s"
            % (subject["subject_id"], year)
        )

    features = {}

    # BASIC
    for pos, pair in natal_pillars.items():
        stem, branch = pair
        onehot(
            features,
            "basic__natal_%s_stem_" % pos,
            stem,
            STEMS,
        )
        onehot(
            features,
            "basic__natal_%s_branch_" % pos,
            branch,
            BRANCHES,
        )

    onehot(
        features,
        "basic__daymaster_stem_",
        day_stem,
        STEMS,
    )
    onehot(
        features,
        "basic__daymaster_element_",
        STEM_ELEMENT_EN[day_stem],
        ELEMENTS,
    )

    for label, stem, branch in [
        ("dw", dw_stem, dw_branch),
        ("sw", sw_stem, sw_branch),
    ]:
        onehot(
            features,
            "basic__%s_stem_" % label,
            stem,
            STEMS,
        )
        onehot(
            features,
            "basic__%s_branch_" % label,
            branch,
            BRANCHES,
        )
        onehot(
            features,
            "basic__%s_element_" % label,
            STEM_ELEMENT_EN[stem],
            ELEMENTS,
        )
        onehot(
            features,
            "basic__%s_yy_" % label,
            STEM_YINYANG[stem],
            ["yang", "yin"],
        )

    natal_element_counts = Counter()

    for stem, branch in natal_pillars.values():
        natal_element_counts[STEM_ELEMENT_EN[stem]] += 1
        natal_element_counts[BRANCH_ELEMENT_EN[branch]] += 1

    for element in ELEMENTS:
        features[
            "basic__natal_element_count_%s" % element
        ] = float(natal_element_counts[element])

    # TENGOD
    for label, stem in [
        ("dw", dw_stem),
        ("sw", sw_stem),
    ]:
        tg = ten_god_group(day_stem, stem)
        onehot(
            features,
            "tengod__%s_group_" % label,
            tg,
            ["peer", "output", "wealth", "officer", "resource"],
        )

    # HIDDEN
    hidden_tg = Counter()
    hidden_element = Counter()

    for _, branch in natal_pillars.values():
        for hidden_stem in HIDDEN_STEMS[branch]:
            hidden_tg[
                ten_god_group(day_stem, hidden_stem)
            ] += 1

            hidden_element[
                STEM_ELEMENT_EN[hidden_stem]
            ] += 1

    for tg in [
        "peer", "output", "wealth", "officer", "resource"
    ]:
        features[
            "hidden__natal_tg_count_%s" % tg
        ] = float(hidden_tg[tg])

    for element in ELEMENTS:
        features[
            "hidden__natal_element_count_%s" % element
        ] = float(hidden_element[element])

    # RELATIONS
    natal_stems = [
        pair[0] for pair in natal_pillars.values()
    ]
    natal_branches = [
        pair[1] for pair in natal_pillars.values()
    ]

    for label, stem, branch in [
        ("dw", dw_stem, dw_branch),
        ("sw", sw_stem, sw_branch),
    ]:
        features[
            "relation__%s_stem_combine_natal" % label
        ] = sum(
            pair_relation(stem, x, STEM_COMBINES)
            for x in natal_stems
        )

        features[
            "relation__%s_stem_clash_natal" % label
        ] = sum(
            pair_relation(stem, x, STEM_CLASHES)
            for x in natal_stems
        )

        features[
            "relation__%s_branch_combine_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_COMBINES)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_clash_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_CLASHES)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_harm_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_HARMS)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_break_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_BREAKS)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_punish_natal" % label
        ] = sum(
            branch_punishment(branch, x)
            for x in natal_branches
        )

    features["relation__sw_dw_stem_combine"] = pair_relation(
        sw_stem, dw_stem, STEM_COMBINES
    )
    features["relation__sw_dw_stem_clash"] = pair_relation(
        sw_stem, dw_stem, STEM_CLASHES
    )
    features["relation__sw_dw_branch_combine"] = pair_relation(
        sw_branch, dw_branch, BRANCH_COMBINES
    )
    features["relation__sw_dw_branch_clash"] = pair_relation(
        sw_branch, dw_branch, BRANCH_CLASHES
    )
    features["relation__sw_dw_branch_harm"] = pair_relation(
        sw_branch, dw_branch, BRANCH_HARMS
    )
    features["relation__sw_dw_branch_break"] = pair_relation(
        sw_branch, dw_branch, BRANCH_BREAKS
    )
    features["relation__sw_dw_branch_punish"] = branch_punishment(
        sw_branch, dw_branch
    )

    # ENGINE STRENGTH
    strength = get_strength_label(result)

    for value in [
        "극약","태약","신약","중화신약",
        "중화신강","신강","태강","극왕"
    ]:
        features[
            "engine__strength_%s" % value
        ] = float(strength == value)

    # V1.3 EXTENDED
    dw_row = get_daewoon_row(daewoon, year)

    if dw_row is None:
        raise ValueError(
            "No active Daewoon row: %s %s"
            % (subject["subject_id"], year)
        )

    features.update(
        yongshin_block_features(
            result, dw_stem, dw_branch, "dw"
        )
    )
    features.update(
        yongshin_block_features(
            result, sw_stem, sw_branch, "sw"
        )
    )

    features.update(
        unseong_block_features(
            result, dw_branch, "dw"
        )
    )
    features.update(
        unseong_block_features(
            result, sw_branch, "sw"
        )
    )

    features.update(
        shinsal_features(dw_row, meta_row)
    )

    features["baseline__control_score"] = (
        control_score_from_meta(meta_row)
    )

    return features

In [11]:
def extract_v4_features(subject, year, result, daewoon, meta_row):
    f = extract_v2_all_features(subject, year, result, daewoon, meta_row)

    natal = result["원국"]
    day_stem, _ = pillar_chars(natal["day"])
    dw_stem, dw_branch = pillar_chars(get_daewoon_pillar(meta_row))
    sw_stem, sw_branch = pillar_chars(get_sewoon_pillar(meta_row, year))

    strength = strength_bucket(result)

    for label, stem, branch in [("dw", dw_stem, dw_branch), ("sw", sw_stem, sw_branch)]:
        tg = exact_ten_god(day_stem, stem)
        onehot(f, "tg10__%s_stem_" % label, tg, EXACT_TENGODS)

        dist = exact_tg_distribution(day_stem, HIDDEN_STEMS[branch])
        for exact_name in EXACT_TENGODS:
            f["tg10__%s_branch_hidden_%s" % (label, exact_name)] = float(dist[exact_name])

        # Contextual interaction: exact dynamic Ten-God activation x natal strength bucket.
        dynamic = {k: 0.5 * (float(tg == k) + float(dist[k])) for k in EXACT_TENGODS}
        for exact_name in EXACT_TENGODS:
            for bucket in ["weak", "neutral", "strong"]:
                f["tgxstrength__%s_%s_%s" % (label, exact_name, bucket)] = (
                    dynamic[exact_name] * float(strength == bucket)
                )

    return f

## 6. First feature generation on the frozen 101 pairs

In [12]:

FEATURE_CACHE = ARTIFACT_DIR / "V4_UNIFIED_DEV_year_features.csv"

required_years = defaultdict(set)
for _, r in pairs.iterrows():
    required_years[r.subject_id].add(int(r.positive_year))
    required_years[r.subject_id].add(int(r.negative_year))

rows = []
for i, sid in enumerate(sorted(required_years)):
    subject = subject_map[sid]
    result, daewoon, meta = compute_v4_subject_engine(subject)

    for year in sorted(required_years[sid]):
        if year not in meta:
            raise RuntimeError("Timeline missing %s %s" % (sid, year))
        features = extract_v4_features(subject, year, result, daewoon, meta[year])
        row = {"subject_id": sid, "year": int(year)}
        row.update(features)
        rows.append(row)

    if (i + 1) % 10 == 0:
        print("engine subjects", i + 1, "/", len(required_years))

year_features = pd.DataFrame(rows)
assert len(year_features) == 202
assert year_features[["subject_id","year"]].drop_duplicates().shape[0] == 202

year_features.to_csv(FEATURE_CACHE, index=False)

feature_manifest = {
    "version": "V4_UNIFIED_DEV_FEATURE_GENERATION_MANIFEST_V1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook_version": NOTEBOOK_VERSION,
    "n_subjects": 101,
    "n_year_rows": int(len(year_features)),
    "n_columns": int(len(year_features.columns)),
    "pair_sha256": sha256_file(PAIR_PATH),
    "saju_engine_sha256": sha256_file(ROOT / "saju_engine.py"),
    "preregistration_spec_sha256": sha256_file(SPEC_PATH),
    "feature_cache_sha256": sha256_file(FEATURE_CACHE),
}
with open(ARTIFACT_DIR / "V4_UNIFIED_DEV_FEATURE_GENERATION_MANIFEST.json","w",encoding="utf-8") as f:
    json.dump(feature_manifest,f,ensure_ascii=False,indent=2)

print("year feature table:", year_features.shape)


[SAJU_DEBUG] original_input: 1969-10-13 16:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1969-10-13
[SAJU_DEBUG] final_datetime(KST): 1969-10-13T16:32:00+09:00
[SAJU_DEBUG] pillars: 연=己酉 월=甲戌 일=辛酉 시=丙申
[SAJU_DEBUG] original_input: 1940-02-03 00:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1940-02-03
[SAJU_DEBUG] final_datetime(KST): 1940-02-03T00:32:00+09:00
[SAJU_DEBUG] pillars: 연=己卯 월=丁丑 일=丙子 시=戊子
[SAJU_DEBUG] original_input: 1958-06-15 20:45
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1958-06-15
[SAJU_DEBUG] final_datetime(KST): 1958-06-15T20:45:00+09:00
[SAJU_DEBUG] pillars: 연=戊戌 월=戊午 일=癸亥 시=壬戌
[SAJU_DEBUG] original_input: 1946-07-15 17:15
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female


engine subjects 10 / 101


[SAJU_DEBUG] original_input: 1924-10-15 16:58
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1924-10-15
[SAJU_DEBUG] final_datetime(KST): 1924-10-15T16:58:00+09:00
[SAJU_DEBUG] pillars: 연=甲子 월=甲戌 일=丁卯 시=戊申
[SAJU_DEBUG] original_input: 1943-03-19 05:05
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1943-03-19
[SAJU_DEBUG] final_datetime(KST): 1943-03-19T05:05:00+09:00
[SAJU_DEBUG] 반시보정: 辛卯→庚寅 (05:05)
[SAJU_DEBUG] pillars: 연=癸未 월=乙卯 일=丙子 시=庚寅
[SAJU_DEBUG] original_input: 1950-03-04 10:01
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1950-03-04
[SAJU_DEBUG] final_datetime(KST): 1950-03-04T10:01:00+09:00
[SAJU_DEBUG] pillars: 연=庚寅 월=戊寅 일=戊戌 시=丁巳
[SAJU_DEBUG] original_input: 1936-01-06 01:15
[SAJU_DEBUG] calendar=solar, is_l

engine subjects 20 / 101


[SAJU_DEBUG] original_input: 1918-02-05 11:54
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1918-02-05
[SAJU_DEBUG] final_datetime(KST): 1918-02-05T11:54:00+09:00
[SAJU_DEBUG] pillars: 연=戊午 월=甲寅 일=癸未 시=戊午
[SAJU_DEBUG] original_input: 1955-02-24 19:05
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1955-02-24
[SAJU_DEBUG] final_datetime(KST): 1955-02-24T19:05:00+09:00
[SAJU_DEBUG] 반시보정: 戊戌→丁酉 (19:05)
[SAJU_DEBUG] pillars: 연=乙未 월=戊寅 일=丙辰 시=丁酉
[SAJU_DEBUG] original_input: 1937-09-02 12:18
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1937-09-02
[SAJU_DEBUG] final_datetime(KST): 1937-09-02T12:18:00+09:00
[SAJU_DEBUG] pillars: 연=丁丑 월=戊申 일=壬辰 시=丙午
[SAJU_DEBUG] original_input: 1936-04-19 10:44
[SAJU_DEBUG] calendar=solar, is_l

...

engine subjects 90 / 101


[SAJU_DEBUG] original_input: 1936-08-29 18:05
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1936-08-29
[SAJU_DEBUG] final_datetime(KST): 1936-08-29T18:05:00+09:00
[SAJU_DEBUG] pillars: 연=丙子 월=丙申 일=癸未 시=辛酉
[SAJU_DEBUG] original_input: 1924-04-12 06:11
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1924-04-12
[SAJU_DEBUG] final_datetime(KST): 1924-04-12T06:11:00+09:00
[SAJU_DEBUG] pillars: 연=甲子 월=戊辰 일=辛酉 시=辛卯
[SAJU_DEBUG] original_input: 1932-11-29 12:09
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1932-11-29
[SAJU_DEBUG] final_datetime(KST): 1932-11-29T12:09:00+09:00
[SAJU_DEBUG] pillars: 연=壬申 월=辛亥 일=甲午 시=庚午
[SAJU_DEBUG] original_input: 1925-11-15 14:21
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

engine subjects 100 / 101


[SAJU_DEBUG] original_input: 1938-04-07 12:24
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1938-04-07
[SAJU_DEBUG] final_datetime(KST): 1938-04-07T12:24:00+09:00
[SAJU_DEBUG] pillars: 연=戊寅 월=丙辰 일=己巳 시=庚午


year feature table: (202, 405)


## 7. Pair-difference table + pre-existing chronology nuisance

In [13]:

meta_cols = {"subject_id", "year"}
feature_cols = [c for c in year_features.columns if c not in meta_cols]
for c in feature_cols:
    year_features[c] = pd.to_numeric(year_features[c], errors="coerce").fillna(0.0)

lookup = year_features.set_index(["subject_id", "year"])
pair_rows = []

for _, r in pairs.iterrows():
    p = lookup.loc[(r.subject_id, int(r.positive_year))]
    n = lookup.loc[(r.subject_id, int(r.negative_year))]

    out = r.to_dict()
    for c in feature_cols:
        out["diff__" + c] = float(p[c] - n[c])

    # Pre-existing V4 nuisance design: lifecycle / chronology x axis.
    age_delta = float(r.positive_age - r.negative_age)
    age_mid = 0.5 * float(r.positive_age + r.negative_age)
    out["nuisance__age_delta"] = age_delta
    out["nuisance__age_delta_x_gap"] = age_delta * float(r.abs_year_gap)
    out["nuisance__age_delta_x_mid"] = age_delta * age_mid

    for axis in sorted(pairs.axis.unique()):
        out["nuisance__age_delta_x_axis__" + axis] = age_delta * float(r.axis == axis)

    pair_rows.append(out)

pair_df = pd.DataFrame(pair_rows)
assert len(pair_df) == 101
assert pair_df.subject_id.nunique() == 101
pair_df.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_pair_diff_feature_table.csv", index=False)

print("pair diff table:", pair_df.shape)


pair diff table: (101, 440)


## 8. Exactly 3 preregistered feature architectures

In [14]:

ALL_DIFF = [c for c in pair_df.columns if c.startswith("diff__")]

TG10_STEM = sorted([
    c for c in ALL_DIFF
    if c.startswith("diff__tg10__") and "_stem_" in c
])
TG10_BRANCH = sorted([
    c for c in ALL_DIFF
    if c.startswith("diff__tg10__") and "_branch_hidden_" in c
])
TG10_SB = sorted(set(TG10_STEM + TG10_BRANCH))

ALL_ASTRO = sorted([
    c for c in ALL_DIFF
    if not c.startswith("diff__baseline__")
])

NUISANCE_AGE_AXIS = sorted([
    c for c in pair_df.columns
    if c.startswith("nuisance__")
])

FEATURE_SETS = {
    "TG10_STEM_BRANCH_L2": TG10_SB,
    "ALL_L2": ALL_ASTRO,
}

assert list(FEATURE_SETS.keys()) + ["ALL_ELASTICNET_AUTO"] == WINNER_ELIGIBLE
assert TG10_SB
assert ALL_ASTRO
assert NUISANCE_AGE_AXIS

print("TG10_STEM_BRANCH_L2:", len(TG10_SB))
print("ALL_L2 / ElasticNet universe:", len(ALL_ASTRO))
print("NUISANCE_AGE_AXIS:", NUISANCE_AGE_AXIS)


TG10_STEM_BRANCH_L2: 40
ALL_L2 / ElasticNet universe: 402
NUISANCE_AGE_AXIS: ['nuisance__age_delta', 'nuisance__age_delta_x_axis__COMPETITIVE', 'nuisance__age_delta_x_axis__PROJECT', 'nuisance__age_delta_x_axis__STATUS', 'nuisance__age_delta_x_gap', 'nuisance__age_delta_x_mid']


## 9. Symmetric pairwise models

In [15]:
def symmetric_xy(frame, columns):
    X = frame[columns].to_numpy(dtype=float)
    y = np.r_[np.ones(len(X), dtype=int), np.zeros(len(X), dtype=int)]
    X2 = np.vstack([X, -X])
    w0 = frame["subject_equal_pair_weight"].to_numpy(dtype=float) / 2.0
    w = np.r_[w0, w0]
    return X2, y, w


def fit_l2(frame, columns, C=0.3):
    X, y, w = symmetric_xy(frame, columns)
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2", C=float(C), solver="liblinear", max_iter=5000,
            random_state=SEED,
        )),
    ])
    model.fit(X, y, model__sample_weight=w)
    return model


def fit_elastic(frame, columns, C, l1_ratio, seed):
    X, y, w = symmetric_xy(frame, columns)
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="elasticnet", solver="saga", C=float(C), l1_ratio=float(l1_ratio),
            max_iter=8000, random_state=int(seed),
        )),
    ])
    model.fit(X, y, model__sample_weight=w)
    return model


def fit_hgb(frame, columns, seed):
    X, y, w = symmetric_xy(frame, columns)
    if HGB_AVAILABLE:
        model = HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=180, max_leaf_nodes=7, max_depth=3,
            min_samples_leaf=10, l2_regularization=1.0, random_state=int(seed),
        )
    else:
        model = GradientBoostingClassifier(
            n_estimators=120, learning_rate=0.04, max_depth=2,
            min_samples_leaf=8, random_state=int(seed),
        )
    model.fit(X, y, sample_weight=w)
    return model


def decision_scores(model, frame, columns):
    X = frame[columns].to_numpy(dtype=float)
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X), dtype=float)
    proba = model.predict_proba(X)[:, 1]
    return np.asarray(proba - 0.5, dtype=float)


def pair_correct_from_score(score, eps=1e-12):
    score = np.asarray(score, dtype=float)
    return np.where(score > eps, 1.0, np.where(score < -eps, 0.0, 0.5))


def subject_macro(frame, correct):
    tmp = frame[["subject_id"]].copy()
    tmp["correct"] = np.asarray(correct, dtype=float)
    return float(tmp.groupby("subject_id")["correct"].mean().mean())


def per_subject(frame, correct):
    tmp = frame[["subject_id"]].copy()
    tmp["correct"] = np.asarray(correct, dtype=float)
    return tmp.groupby("subject_id")["correct"].mean()

## 10. Subject folds balanced on axis + chronology direction

In [19]:
from sklearn.model_selection import StratifiedKFold

def make_subject_folds(frame, n_folds, seed):
    """
    One subject = one pair in this frozen corpus.

    Stratify jointly on:
      - axis
      - positive_earlier chronology direction

    This guarantees non-empty folds and directly controls the
    axis-specific chronology imbalance.
    """
    meta = frame[
        ["subject_id", "axis", "positive_earlier_calc"]
    ].copy()

    # Frozen unified DEV assumption: exactly one pair per subject.
    assert meta["subject_id"].nunique() == len(meta)

    meta["stratum"] = (
        meta["axis"].astype(str)
        + "__"
        + meta["positive_earlier_calc"].astype(int).astype(str)
    )

    stratum_counts = meta["stratum"].value_counts()

    if int(stratum_counts.min()) < int(n_folds):
        raise RuntimeError(
            "Cannot stratify %d folds: smallest axis×chronology stratum has only %d subjects.\n%s"
            % (n_folds, int(stratum_counts.min()), stratum_counts.to_string())
        )

    splitter = StratifiedKFold(
        n_splits=int(n_folds),
        shuffle=True,
        random_state=int(seed),
    )

    X_dummy = np.zeros((len(meta), 1))
    y = meta["stratum"].to_numpy()

    folds = []

    for _, test_idx in splitter.split(X_dummy, y):
        ids = meta.iloc[test_idx]["subject_id"].tolist()
        assert len(ids) > 0
        folds.append(ids)

    assert len(folds) == int(n_folds)
    assert sum(len(x) for x in folds) == len(meta)

    # Every subject must appear in exactly one test fold.
    flat = [sid for fold in folds for sid in fold]
    assert len(flat) == len(set(flat)) == len(meta)

    return folds


# Audit one split before modeling.
audit_folds = make_subject_folds(pair_df, OUTER_FOLDS, SEED)

fold_audit = []

for i, ids in enumerate(audit_folds):
    x = pair_df[pair_df.subject_id.isin(ids)]

    assert len(x) > 0

    fold_audit.append({
        "fold": i,
        "n": len(x),
        "positive_earlier_share": float(x.positive_earlier_calc.mean()),
        **{
            "n_" + a: int((x.axis == a).sum())
            for a in sorted(pair_df.axis.unique())
        }
    })

fold_audit = pd.DataFrame(fold_audit)

fold_audit.to_csv(
    ARTIFACT_DIR / "V4_UNIFIED_DEV_fold_balance_audit.csv",
    index=False
)

display(fold_audit)

,fold,n,positive_earlier_share,n_COMPETITIVE,n_PROJECT,n_STATUS
0,0,21,0.571429,7,5,9
1,1,20,0.550000,6,5,9
2,2,20,0.600000,6,5,9
3,3,20,0.550000,6,5,9
4,4,20,0.550000,6,5,9


## 11. Nested ElasticNet tuning

In [20]:
def tune_elastic(train_frame, columns, seed):
    folds = make_subject_folds(train_frame, min(INNER_FOLDS, train_frame.subject_id.nunique()), seed)
    rows = []

    for C in ELASTIC_C_GRID:
        for l1 in ELASTIC_L1_GRID:
            scores = []
            for fold_i, test_subjects in enumerate(folds):
                tr = train_frame[~train_frame.subject_id.isin(test_subjects)]
                te = train_frame[train_frame.subject_id.isin(test_subjects)]
                if tr.subject_id.nunique() < 5 or te.empty:
                    continue
                model = fit_elastic(tr, columns, C, l1, seed + fold_i)
                correct = pair_correct_from_score(decision_scores(model, te, columns))
                scores.append(subject_macro(te, correct))
            rows.append({
                "C": C, "l1_ratio": l1,
                "inner_macro": float(np.mean(scores)) if scores else np.nan,
                "inner_p10": float(np.quantile(scores, .10)) if scores else np.nan,
            })

    grid = pd.DataFrame(rows).sort_values(
        ["inner_macro", "inner_p10", "C"], ascending=[False, False, True]
    ).reset_index(drop=True)
    return grid.iloc[0].to_dict(), grid

## 12. Repeated nested subject-CV tournament

In [21]:

ASTRO_MODELS = WINNER_ELIGIBLE
ALL_MODELS = [
    "AGE_YOUNGER_FIXED",
    "AGE_LATER_FIXED",
    "NUISANCE_AGE_AXIS",
    "CONTROL_FIXED",
] + ASTRO_MODELS

CONTROL_COL = "diff__baseline__control_score"
assert CONTROL_COL in pair_df.columns

outer_pair_rows = []
outer_subject_rows = []
elastic_grid_rows = []
elastic_selection_rows = []

for repeat in range(OUTER_REPEATS):
    folds = make_subject_folds(pair_df, OUTER_FOLDS, SEED + 1000 * repeat)

    for fold_i, test_subjects in enumerate(folds):
        train = pair_df[~pair_df.subject_id.isin(test_subjects)].copy()
        test = pair_df[pair_df.subject_id.isin(test_subjects)].copy()
        assert set(train.subject_id).isdisjoint(set(test.subject_id))

        models = {}

        models["TG10_STEM_BRANCH_L2"] = (
            fit_l2(train, TG10_SB, C=0.3), TG10_SB
        )
        models["ALL_L2"] = (
            fit_l2(train, ALL_ASTRO, C=0.3), ALL_ASTRO
        )
        models["NUISANCE_AGE_AXIS"] = (
            fit_l2(train, NUISANCE_AGE_AXIS, C=0.3), NUISANCE_AGE_AXIS
        )

        best_elastic, grid = tune_elastic(
            train, ALL_ASTRO, SEED + 10000*repeat + fold_i
        )
        grid["repeat"] = repeat
        grid["fold"] = fold_i
        elastic_grid_rows.append(grid)

        elastic_model = fit_elastic(
            train, ALL_ASTRO,
            best_elastic["C"], best_elastic["l1_ratio"],
            SEED + 20000*repeat + fold_i,
        )
        models["ALL_ELASTICNET_AUTO"] = (elastic_model, ALL_ASTRO)

        coef = elastic_model.named_steps["model"].coef_.ravel()
        for col, value in zip(ALL_ASTRO, coef):
            elastic_selection_rows.append({
                "repeat": repeat,
                "fold": fold_i,
                "feature": col,
                "selected": float(abs(value)>1e-8),
                "coefficient": float(value),
            })

        fixed = {
            "AGE_YOUNGER_FIXED": test["positive_earlier_calc"].to_numpy(dtype=float),
            "AGE_LATER_FIXED": 1.0 - test["positive_earlier_calc"].to_numpy(dtype=float),
            "CONTROL_FIXED": pair_correct_from_score(
                test[CONTROL_COL].to_numpy(dtype=float)
            ),
        }

        scored = {k: np.asarray(v,dtype=float) for k,v in fixed.items()}
        for name, (model, cols) in models.items():
            scored[name] = pair_correct_from_score(
                decision_scores(model, test, cols)
            )

        for name, correct in scored.items():
            tmp = test[["pair_id","subject_id","axis","positive_earlier_calc"]].copy()
            tmp["repeat"] = repeat
            tmp["fold"] = fold_i
            tmp["model"] = name
            tmp["correct"] = correct
            outer_pair_rows.append(tmp)

            sm = tmp.groupby("subject_id")["correct"].mean().reset_index()
            sm["repeat"] = repeat
            sm["fold"] = fold_i
            sm["model"] = name
            outer_subject_rows.append(sm)

outer_pair = pd.concat(outer_pair_rows, ignore_index=True)
outer_subject = pd.concat(outer_subject_rows, ignore_index=True)
elastic_grid = pd.concat(elastic_grid_rows, ignore_index=True)
elastic_selection = pd.DataFrame(elastic_selection_rows)

outer_pair.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_OOF_pair_scores.csv", index=False)
outer_subject.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_OOF_subject_scores.csv", index=False)
elastic_grid.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_elasticnet_inner_grid.csv", index=False)
elastic_selection.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_elasticnet_feature_selection_by_fold.csv", index=False)

print("Nested tournament complete")


Nested tournament complete


## 13. Overall leaderboard

In [22]:

repeat_metrics = (
    outer_subject.groupby(["model","repeat"])["correct"].mean()
    .reset_index(name="subject_macro")
)

leader_rows = []
for model, g in repeat_metrics.groupby("model"):
    vals = g.subject_macro.to_numpy(dtype=float)
    leader_rows.append({
        "model": model,
        "pairwise_mean": float(vals.mean()),
        "pairwise_std": float(vals.std(ddof=1)) if len(vals)>1 else 0.0,
        "pairwise_p10": float(np.quantile(vals,.10)),
        "pairwise_min": float(vals.min()),
    })

leaderboard = pd.DataFrame(leader_rows).sort_values(
    "pairwise_mean", ascending=False
).reset_index(drop=True)
leaderboard.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_nested_leaderboard.csv", index=False)
display(leaderboard)


,model,pairwise_mean,pairwise_std,pairwise_p10,pairwise_min
0,NUISANCE_AGE_AXIS,0.662376,3.130968e-03,0.662376,0.653465
1,AGE_YOUNGER_FIXED,0.564356,1.170278e-16,0.564356,0.564356
2,CONTROL_FIXED,0.554455,1.170278e-16,0.554455,0.554455
3,ALL_ELASTICNET_AUTO,0.516832,1.618710e-01,0.267327,0.267327
4,ALL_L2,0.486139,3.598722e-02,0.441584,0.405941
5,TG10_STEM_BRANCH_L2,0.477228,2.907289e-02,0.453465,0.435644
6,AGE_LATER_FIXED,0.435644,5.851389e-17,0.435644,0.435644


## 14. Mandatory axis diagnostics

In [23]:

pair_avg = (
    outer_pair.groupby(["model","pair_id","subject_id","axis"])["correct"]
    .mean().reset_index()
)

axis_rows = []
for (model, axis), g in pair_avg.groupby(["model","axis"]):
    axis_rows.append({
        "model": model,
        "axis": axis,
        "n_pairs": len(g),
        "subject_macro_pairwise": float(g.correct.mean()),
    })

axis_df = pd.DataFrame(axis_rows)
axis_df.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_axis_diagnostics.csv", index=False)
display(axis_df.pivot(index="model",columns="axis",values="subject_macro_pairwise"))


axis,COMPETITIVE,PROJECT,STATUS
model,,,
AGE_LATER_FIXED,0.419355,0.800,0.244444
AGE_YOUNGER_FIXED,0.580645,0.200,0.755556
ALL_ELASTICNET_AUTO,0.490323,0.592,0.493333
ALL_L2,0.429032,0.648,0.435556
CONTROL_FIXED,0.548387,0.620,0.522222
NUISANCE_AGE_AXIS,0.416129,0.800,0.755556
TG10_STEM_BRANCH_L2,0.409677,0.556,0.480000


## 15. Subject bootstrap versus chronology nuisance and Production Control

In [24]:

subject_avg = (
    outer_subject.groupby(["model","subject_id"])["correct"].mean().reset_index()
)
wide = subject_avg.pivot(index="subject_id",columns="model",values="correct")

rng = np.random.RandomState(SEED + 999)
boot_rows = []

for model in WINNER_ELIGIBLE:
    for reference in ["NUISANCE_AGE_AXIS","CONTROL_FIXED"]:
        shared = wide[[model,reference]].dropna()
        ids = shared.index.to_numpy()
        deltas = []

        for _ in range(N_BOOTSTRAP):
            sample = rng.choice(ids, size=len(ids), replace=True)
            d = (
                shared.loc[sample,model].to_numpy()
                - shared.loc[sample,reference].to_numpy()
            ).mean()
            deltas.append(float(d))

        arr = np.asarray(deltas)
        observed = float((shared[model]-shared[reference]).mean())

        boot_rows.append({
            "model": model,
            "reference": reference,
            "observed_delta": observed,
            "bootstrap_mean_delta": float(arr.mean()),
            "ci025": float(np.quantile(arr,.025)),
            "ci975": float(np.quantile(arr,.975)),
            "p_delta_gt_0": float((arr>0).mean()),
        })

boot = pd.DataFrame(boot_rows)
boot.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_model_reference_bootstrap.csv", index=False)
display(boot)


,model,reference,observed_delta,bootstrap_mean_delta,ci025,ci975,p_delta_gt_0
0,TG10_STEM_BRANCH_L2,NUISANCE_AGE_AXIS,-0.185149,-0.184965,-0.298045,-0.071287,0.00085
1,TG10_STEM_BRANCH_L2,CONTROL_FIXED,-0.077228,-0.077242,-0.191089,0.036634,0.09240
2,ALL_L2,NUISANCE_AGE_AXIS,-0.176238,-0.175857,-0.290099,-0.060396,0.00120
3,ALL_L2,CONTROL_FIXED,-0.068317,-0.068631,-0.172302,0.034653,0.09965
4,ALL_ELASTICNET_AUTO,NUISANCE_AGE_AXIS,-0.145545,-0.144833,-0.242574,-0.047525,0.00215
5,ALL_ELASTICNET_AUTO,CONTROL_FIXED,-0.037624,-0.037506,-0.131683,0.057426,0.21915


## 16. ElasticNet stability

In [25]:

selection_summary = (
    elastic_selection.groupby("feature")
    .agg(
        selection_frequency=("selected","mean"),
        mean_coefficient=("coefficient","mean")
    )
    .reset_index()
    .sort_values(["selection_frequency","feature"],ascending=[False,True])
)
selection_summary.to_csv(
    ARTIFACT_DIR / "V4_UNIFIED_DEV_elasticnet_selection_frequency.csv",
    index=False
)
display(selection_summary.head(100))


,feature,selection_frequency,mean_coefficient
17,diff__basic__dw_branch_午,0.40,0.130273
38,diff__basic__dw_stem_戊,0.36,0.056365
396,diff__yongshin__sw_hee_fit,0.32,0.070671
148,diff__basic__sw_branch_酉,0.30,0.108044
157,diff__basic__sw_stem_壬,0.30,-0.102259
...,...,...,...
382,diff__unseong__sw_state_장생,0.12,-0.007827
22,diff__basic__dw_branch_戌,0.10,-0.021321
30,diff__basic__dw_element_water,0.10,0.009739
32,diff__basic__dw_stem_丁,0.10,-0.018566


## 17. Preregistered promotion gates

A model survives only if it satisfies all numeric gates frozen in 11:

1. overall repeated outer-CV subject-macro ≥ 0.55  
2. ≥ +1%p versus `CONTROL_FIXED`  
3. bootstrap P(delta>0) ≥ 0.80 versus Control  
4. ≥ +1%p versus `NUISANCE_AGE_AXIS`  
5. bootstrap P(delta>0) ≥ 0.80 versus nuisance  
6. repeated-CV p10 ≥ 0.48  

Axis results are mandatory diagnostics. No new numerical axis threshold is invented after seeing results.


In [26]:

lead = leaderboard.set_index("model")
boot_lookup = boot.set_index(["model","reference"])

gate_rows = []
for model in WINNER_ELIGIBLE:
    overall = float(lead.loc[model,"pairwise_mean"])
    p10 = float(lead.loc[model,"pairwise_p10"])

    d_n = float(boot_lookup.loc[(model,"NUISANCE_AGE_AXIS"),"observed_delta"])
    p_n = float(boot_lookup.loc[(model,"NUISANCE_AGE_AXIS"),"p_delta_gt_0"])
    d_c = float(boot_lookup.loc[(model,"CONTROL_FIXED"),"observed_delta"])
    p_c = float(boot_lookup.loc[(model,"CONTROL_FIXED"),"p_delta_gt_0"])

    axis_values = axis_df[axis_df.model==model].subject_macro_pairwise.to_numpy(dtype=float)
    axis_min = float(axis_values.min())

    passes = {
        "overall_ge_055": overall >= MIN_OVERALL_MACRO,
        "delta_nuisance_ge_001": d_n >= MIN_DELTA_VS_REFERENCE,
        "boot_nuisance_ge_080": p_n >= MIN_BOOT_P_DELTA_GT_0,
        "delta_control_ge_001": d_c >= MIN_DELTA_VS_REFERENCE,
        "boot_control_ge_080": p_c >= MIN_BOOT_P_DELTA_GT_0,
        "p10_ge_048": p10 >= MIN_P10,
    }

    gate_rows.append({
        "model": model,
        "overall": overall,
        "p10": p10,
        "delta_vs_nuisance": d_n,
        "p_vs_nuisance": p_n,
        "delta_vs_control": d_c,
        "p_vs_control": p_c,
        "axis_min_diagnostic": axis_min,
        **passes,
        "all_numeric_gates": all(passes.values()),
    })

gates = pd.DataFrame(gate_rows).sort_values(
    ["all_numeric_gates","overall","delta_vs_control","p10"],
    ascending=[False,False,False,False]
)
gates.to_csv(ARTIFACT_DIR / "V4_UNIFIED_DEV_candidate_gates.csv", index=False)
display(gates)

survivors = gates[gates.all_numeric_gates].copy()

if len(survivors):
    # Preregistered tie-break order already matches this sorting.
    winner = survivors.iloc[0].model
    status = "V4_UNIFIED_DEV_DISCOVERY_ARCHITECTURE_FROZEN_FOR_NEW_GOLD_HOLDOUT"
    reason = "At least one preregistered architecture passed all frozen numeric discovery gates."
else:
    winner = None
    status = "V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE"
    reason = "No preregistered astrology architecture passed all frozen discovery gates."

print(status, winner)


,model,overall,p10,delta_vs_nuisance,p_vs_nuisance,delta_vs_control,p_vs_control,axis_min_diagnostic,overall_ge_055,delta_nuisance_ge_001,boot_nuisance_ge_080,delta_control_ge_001,boot_control_ge_080,p10_ge_048,all_numeric_gates
2,ALL_ELASTICNET_AUTO,0.516832,0.267327,-0.145545,0.00215,-0.037624,0.21915,0.490323,False,False,False,False,False,False,False
1,ALL_L2,0.486139,0.441584,-0.176238,0.00120,-0.068317,0.09965,0.429032,False,False,False,False,False,False,False
0,TG10_STEM_BRANCH_L2,0.477228,0.453465,-0.185149,0.00085,-0.077228,0.09240,0.409677,False,False,False,False,False,False,False


V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE None


## 18. Immutable tournament decision

In [27]:

decision = {
    "version": "V4_UNIFIED_DEV_3ARCH_TOURNAMENT_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": status,
    "winner": winner,
    "reason": reason,
    "corpus": {
        "n_pairs": 101,
        "n_subjects": 101,
        "axis_counts": {k:int(v) for k,v in pairs.groupby("axis").size().to_dict().items()},
        "pair_sha256": sha256_file(PAIR_PATH),
        "positive_earlier_share": float(pairs.positive_earlier_calc.mean()),
    },
    "engine": {
        "saju_engine_sha256": sha256_file(ROOT / "saju_engine.py"),
    },
    "winner_eligible_architectures": WINNER_ELIGIBLE,
    "leaderboard": leaderboard.to_dict(orient="records"),
    "gates": gates.to_dict(orient="records"),
    "axis_diagnostics": axis_df.to_dict(orient="records"),
    "holdout_integrity": {
        "NEW_CONFIRM_loaded": False,
        "Validation_B_loaded": False,
        "Public_CHECK_loaded": False,
        "Public_FINAL_loaded": False,
    },
    "next_rule": (
        "If a winner is frozen, do not tune it again on this 101-pair corpus. "
        "Construct a completely new GOLD holdout under one frozen prospective collection protocol "
        "and score the frozen winner and Production Control once. "
        "If no winner survives, do not open a holdout and do not create a post-hoc fourth candidate."
    )
}

with open(ARTIFACT_DIR / "V4_UNIFIED_DEV_3ARCH_TOURNAMENT_DECISION.json","w",encoding="utf-8") as f:
    json.dump(decision,f,ensure_ascii=False,indent=2)

print(json.dumps({
    "status": status,
    "winner": winner,
    "next_rule": decision["next_rule"],
}, ensure_ascii=False, indent=2))


{
  "status": "V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE",
  "winner": null,
  "next_rule": "If a winner is frozen, do not tune it again on this 101-pair corpus. Construct a completely new GOLD holdout under one frozen prospective collection protocol and score the frozen winner and Production Control once. If no winner survives, do not open a holdout and do not create a post-hoc fourth candidate."
}


## What to send back

After **Kernel Restart → Run All**, send:

```text
V4_UNIFIED_DEV_3ARCH_TOURNAMENT_DECISION.json
V4_UNIFIED_DEV_nested_leaderboard.csv
V4_UNIFIED_DEV_candidate_gates.csv
V4_UNIFIED_DEV_axis_diagnostics.csv
V4_UNIFIED_DEV_model_reference_bootstrap.csv
V4_UNIFIED_DEV_elasticnet_selection_frequency.csv
V4_UNIFIED_DEV_FEATURE_GENERATION_MANIFEST.json
```

Do **not** open any sealed holdout after the run unless the decision explicitly freezes a discovery winner.
